# AI-Driven Digital Twin for Water Distribution Networks
## Chapter 6: Urban Water Demand Forecasting Using the Battle of Water Demand Forecasting (BWDF) Dataset

**Team:** Azim Abdulla, Adithya Shaji, Noel John  
**Supervisor:** Dr. Archana T | School of Computer Science & Engineering (SCOPE), VIT  

---

### Objective
This notebook demonstrates the end-to-end Time-Series Machine Learning pipeline for short-term urban water demand forecasting:
1. **Dataset Ingestion:** 19,683 hourly observations (Jan 2021 – Mar 2023) from the official **Battle of Water Demand Forecasting (BWDF)** benchmark.
2. **Feature Engineering:** Autoregressive Lags ($1\text{h}, 24\text{h}, 168\text{h}$), Rolling Window Statistics, and Cyclical Time Encoding.
3. **Chronological Model Training:** `HistGradientBoostingRegressor` trained with strict temporal split.
4. **Multi-Step 24-Hour Forecasting & Digital Twin Distribution** across network junction nodes.

### 1. Setup & Library Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Paths
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd(), 'data')
CLEANED_CSV = os.path.join(DATA_DIR, 'bwdf_demand_cleaned.csv')
MODEL_FILE = os.path.join(DATA_DIR, 'demand_forecast_model.joblib')

print("Environment initialized. Loading cleaned BWDF dataset...")

### 2. Dataset Overview: 19,683 Hourly Records
Inspect the merged inflow rates ($L/s$) across 10 District Metered Areas and correlated meteorological data.

In [ ]:
df = pd.read_csv(CLEANED_CSV)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Dataset Shape: {df.shape} ({df['timestamp'].min()} to {df['timestamp'].max()})")
print("\nSummary Statistics (Demand in L/s):")
display(df[['total_demand_lps', 'dma_1_lps', 'dma_2_lps', 'temperature_c', 'rainfall_mm']].describe())

### 3. Exploratory Data Analysis (EDA): Diurnal & Weekly Demand Profiles
Visualize the 24-hour daily rhythm and weekend shift in water consumption.

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['day_name'] = df['timestamp'].dt.day_name()
df['is_weekend'] = df['timestamp'].dt.dayofweek >= 5

plt.figure(figsize=(12, 5))
sns.lineplot(data=df, x='hour', y='total_demand_lps', hue='is_weekend', palette=['#0284c7', '#f59e0b'], errorbar='sd')
plt.title('Average 24-Hour Diurnal Water Demand Curve: Weekday vs. Weekend (L/s)')
plt.xlabel('Hour of the Day (0-23)')
plt.ylabel('Total Net Inflow (L/s)')
plt.legend(['Weekday (Mon-Fri)', 'Weekend (Sat-Sun)'])
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

### 4. Feature Engineering: Autoregressive Lags & Rolling Statistics

In [ ]:
from demand_forecast.feature_engineering import build_demand_features

df_feat, feature_cols = build_demand_features(df, target_col='total_demand_lps')
print(f"Extracted {len(feature_cols)} time-series features:")
print(feature_cols)
df_feat.head(3)

### 5. Chronological Model Training (80/20 Temporal Split)
We train a `HistGradientBoostingRegressor` to predict future demand without time-leakage.

In [ ]:
split_idx = int(len(df_feat) * 0.80)
train_df = df_feat.iloc[:split_idx]
test_df = df_feat.iloc[split_idx:]

X_train = train_df[feature_cols]
y_train = train_df['total_demand_lps']
X_test = test_df[feature_cols]
y_test = test_df['total_demand_lps']

model = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.08, max_depth=8, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / np.maximum(y_test, 1e-3))) * 100.0

print(f"Test Partition R² Score: {r2 * 100:.2f}%")
print(f"Mean Absolute Error:     {mae:.2f} L/s")
print(f"Root Mean Squared Error: {rmse:.2f} L/s")
print(f"MAPE:                    {mape:.2f}%")

### 6. Forecast vs. Actual Ground Truth Comparison
Plot a 1-week test window showing true measured consumption vs. the ML predicted forecast curve.

In [ ]:
# Plot 1-week sample (168 hours)
sample_test = test_df.iloc[200:368].copy()
sample_pred = y_pred[200:368]

plt.figure(figsize=(14, 6))
plt.plot(sample_test['timestamp'], sample_test['total_demand_lps'], label='Actual Measured Flow (L/s)', color='#0ea5e9', linewidth=2)
plt.plot(sample_test['timestamp'], sample_pred, label='ML Forecast (HistGradientBoosting)', color='#f43f5e', linestyle='--', linewidth=2)
plt.fill_between(sample_test['timestamp'], sample_pred - 1.96 * rmse, sample_pred + 1.96 * rmse, color='#f43f5e', alpha=0.15, label='95% Confidence Interval')

plt.title('7-Day Municipal Water Demand Forecast vs. Ground Truth Observations (L/s)')
plt.xlabel('Timestamp')
plt.ylabel('Demand Rate (L/s)')
plt.legend()
plt.tight_layout()
plt.show()

### 7. Real-Time Digital Twin 24-Hour Multi-Step Forecast
Demonstrates the inference function that feeds forecasted demands into the digital twin.

In [ ]:
from demand_forecast.inference import generate_24h_demand_forecast

forecast_res = generate_24h_demand_forecast(base_temperature=26.0, is_weekend=0)
print("\n--- 24-Hour Demand Forecast Output ---")
print(f"Source: {forecast_res['dataset_source']}")
print(f"Peak Demand: {forecast_res['peak_demand']['hour']} -> {forecast_res['peak_demand']['value_lps']} L/s")
print(f"Minimum Demand: {forecast_res['minimum_demand']['hour']} -> {forecast_res['minimum_demand']['value_lps']} L/s")

forecast_df = pd.DataFrame(forecast_res['forecast_24h'])
display(forecast_df[['hour', 'forecast_demand_lps', 'lower_bound_lps', 'upper_bound_lps', 'temperature_c', 'is_peak']].head(6))